# WWF Finance Tool - Aggregation of ready made Temperature Scores
This notebook can be used to run portfolio aggregation using files with ready made Temperature Scores from data providers. See notebooks #1 and #4 for more details on portfolio aggregation. 

Please see the [methodology](https://sciencebasedtargets.org/wp-content/uploads/2020/09/Temperature-Rating-Methodology-V1.pdf), [guidance](https://sciencebasedtargets.org/wp-content/uploads/2020/10/Financial-Sector-Science-Based-Targets-Guidance-Pilot-Version.pdf) and the [technical documentation](https://wwf-sweden.github.io/ITR-tool/)  for more details on the different aggregation methods.

See 1_analysis_example (on [Colab](https://colab.research.google.com/github/WWF-Sweden/ITR-tool/blob/main/examples/1_analysis_example.ipynb) or [Github](https://github.com/WWF-Sweden/ITR-tool/blob/main/examples/1_analysis_example.ipynb)) for more in depth example of how to work with Jupyter Notebooks in general and WWF notebooks in particular. 


## Setting up
First we will set up the imports and load the file with the company data and temperature scores. 


In [ ]:
%pip install wwf-itr

In [ ]:
import ITR
from ITR.portfolio_aggregation import PortfolioAggregationMethod
from ITR.temperature_score import TemperatureScore
from ITR.interfaces import ETimeFrames, EScope
import pandas as pd
import urllib.request
import os

from ITR.example_utils import collect_company_contributions, plot_grouped_statistics, anonymize, print_aggregations, \
    plot_grouped_heatmap, print_scenario_gain, print_grouped_scores, get_contributions_per_group

In [ ]:
# Download the dummy data
import urllib.request
import os

if not os.path.isdir("data"):
    os.mkdir("data")
if not os.path.isfile("data/example_data_provider_TS.xlsx"):
    urllib.request.urlretrieve("https://github.com/WWF-Sweden/ITR-tool/raw/main/examples/data/example_data_provider_TS.xlsx", "data/example_data_provider_TS.xlsx")


In [ ]:
# Create a temperature score instance with the desired time frames and scopes
ts = TemperatureScore(time_frames=[ETimeFrames.SHORT, ETimeFrames.MID, ETimeFrames.LONG], 
    scopes=[EScope.S1, EScope.S2, EScope.S1S2, EScope.S3, EScope.S1S2S3])

### Load the temperature data

In [ ]:
df_portfolio = pd.read_excel("data/example_data_provider_TS.xlsx")
# Convert the scope and time_frame columns to enums (EScope and ETimeFrames)
df_portfolio['scope'] = df_portfolio['scope'].apply(lambda x: EScope[x.upper()])
df_portfolio['time_frame'] = df_portfolio['time_frame'].apply(lambda x: ETimeFrames[x.upper()])
if not 'temperature_results' in df_portfolio.columns:
    df_portfolio['temperature_results'] = 0
# Create a dictionary to store the portfolio scores for different aggregation methods
scores_collection = {}

## Calculate the aggregated temperature score
Calculate an aggregated temperature score. This can be done using different aggregation methods. The termperature scores are calculated per time-frame/scope combination.

### WATS
Weighted Average Temperature Score (WATS): Temperature scores are allocated based on portfolio weights.
This method uses the "investment_value" field to be defined in your portfolio data.

In [ ]:
ts.aggregation_method = PortfolioAggregationMethod.WATS  
try:
    aggregated_scores = ts.aggregate_scores(df_portfolio)
    df_wats = pd.DataFrame(aggregated_scores.model_dump()).apply(lambda x: x.map(lambda y: round(y['all']['score'], 2) 
                if y is not None and y['all'] is not None and 'score' in y['all'] else None))
    scores_collection.update({'WATS': df_wats})
    display(df_wats)
except ValueError as e:
    print(f"Could not calculate WATS scores — missing data:\n{e}")


### TETS
Total emissions weighted temperature score (TETS): Temperature scores are allocated based on historical emission weights using total company emissions. 
In addition to the portfolios "investment value" the TETS method requires company emissions, please refer to [Data Legends - Fundamental Data](https://wwf-sweden.github.io/ITR-tool/Legends.html#fundamental-data) for more details

In [ ]:
ts.aggregation_method = PortfolioAggregationMethod.TETS 
try:
    aggregated_scores = ts.aggregate_scores(df_portfolio)
    df_tets = pd.DataFrame(aggregated_scores.model_dump()).apply(lambda x: x.map(lambda y: round(y['all']['score'], 2) 
                if y is not None and y['all'] is not None and 'score' in y['all'] else None))
    scores_collection.update({'TETS': df_tets})
    display(df_tets)
except ValueError as e:
    print(f"Could not calculate TETS scores — missing data:\n{e}")

### MOTS
Market Owned emissions weighted temperature score (MOTS): Temperature scores are allocated based on an equity ownership approach.
In addition to the portfolios "investment value" the MOTS method requires company emissions and market cap, please refer to  [Data Legends - Fundamental Data](https://wwf-sweden.github.io/ITR-tool/Legends.html#fundamental-data) for more details

In [ ]:
ts.aggregation_method = PortfolioAggregationMethod.MOTS 
try:
    aggregated_scores = ts.aggregate_scores(df_portfolio)
    df_mots = pd.DataFrame(aggregated_scores.model_dump()).apply(lambda x: x.map(lambda y: round(y['all']['score'], 2) 
                if y is not None and y['all'] is not None and 'score' in y['all'] else None))
    scores_collection.update({'MOTS': df_mots})
    display(df_mots)
except ValueError as e:
    print(f"Could not calculate MOTS scores — missing data:\n{e}")

### EOTS
Enterprise Owned emissions weighted temperature score (EOTS): Temperature scores are allocated based
on an enterprise ownership approach. 
In addition to the portfolios "investment value" the EOTS method requires company emissions and enterprise value, please refer to  [Data Legends - Fundamental Data](https://wwf-sweden.github.io/ITR-tool/Legends.html#fundamental-data) for more details

In [ ]:
ts.aggregation_method = PortfolioAggregationMethod.EOTS 
try:
    aggregated_scores = ts.aggregate_scores(df_portfolio)
    df_eots = pd.DataFrame(aggregated_scores.model_dump()).apply(lambda x: x.map(lambda y: round(y['all']['score'], 2) 
                if y is not None and y['all'] is not None and 'score' in y['all'] else None))
    scores_collection.update({'EOTS': df_eots})
    display(df_eots)
except ValueError as e:
    print(f"Could not calculate EOTS scores — missing data:\n{e}")

### ECOTS
Enterprise Value + Cash emissions weighted temperature score (ECOTS): Temperature scores are allocated based on an enterprise value (EV) plus cash & equivalents ownership approach. 
In addition to the portfolios "investment value" the ECOTS method requires company emissions, company cash equivalents and enterprise value; please refer to  [Data Legends - Fundamental Data](https://wwf-sweden.github.io/ITR-tool/Legends.html#fundamental-data) for more details

In [ ]:
ts.aggregation_method = PortfolioAggregationMethod.ECOTS 
try:
    aggregated_scores = ts.aggregate_scores(df_portfolio)
    df_ecots = pd.DataFrame(aggregated_scores.model_dump()).apply(lambda x: x.map(lambda y: round(y['all']['score'], 2) 
                if y is not None and y['all'] is not None and 'score' in y['all'] else None))
    scores_collection.update({'ECOTS': df_ecots})
    display(df_ecots)
except ValueError as e:
    print(f"Could not calculate ECOTS scores — missing data:\n{e}")

### AOTS
Total Assets emissions weighted temperature score (AOTS): Temperature scores are allocated based on a total assets ownership approach. 
In addition to the portfolios "investment value" the AOTS method requires company emissions and company total assets; please refer to  [Data Legends - Fundamental Data](https://wwf-sweden.github.io/ITR-tool/Legends.html#fundamental-data) for more details

In [ ]:
ts.aggregation_method = PortfolioAggregationMethod.AOTS  
try:
    aggregated_scores = ts.aggregate_scores(df_portfolio)
    df_aots = pd.DataFrame(aggregated_scores.model_dump()).apply(lambda x: x.map(lambda y: round(y['all']['score'], 2) 
                if y is not None and y['all'] is not None and 'score' in y['all'] else None))
    scores_collection.update({'AOTS': df_aots})
    display(df_aots)
except ValueError as e:
    print(f"Could not calculate AOTS scores — missing data:\n{e}")

### ROTS
Revenue owned emissions weighted temperature score (ROTS): Temperature scores are allocated based on the share of revenue.
In addition to the portfolios "investment value" the ROTS method requires company emissions and company revenue; please refer to  [Data Legends - Fundamental Data](https://wwf-sweden.github.io/ITR-tool/Legends.html#fundamental-data) for more details

In [ ]:
ts.aggregation_method = PortfolioAggregationMethod.ROTS
try:
    aggregated_scores = ts.aggregate_scores(df_portfolio)
    df_rots = pd.DataFrame(aggregated_scores.model_dump()).apply(lambda x: x.map(lambda y: round(y['all']['score'], 2) 
                if y is not None and y['all'] is not None and 'score' in y['all'] else None))
    scores_collection.update({'ROTS': df_rots})
    display(df_rots)
except ValueError as e:
    print(f"Could not calculate ROTS scores — missing data:\n{e}")

See below how each aggregation method impact the scores on for each time frame and scope combination

In [ ]:
pd.concat(scores_collection, axis=0)

### Generate a heat map over the temperature scores per sector and region

The cell below creates a single general `TemperatureScore` instance used for all grouped aggregations. Configure the following before running:

- **`time_frames`** — Set broadly upfront (`SHORT`, `MID`, `LONG`). Cannot be changed without reinstantiating, as changing them would invalidate already-computed results.
- **`scopes`** — Same constraint as `time_frames`. Set to all scopes you intend to analyse. Only scopes listed here are available in `analysis_parameters` below — referencing an excluded scope raises a `TypeError`.
- **`aggregation_method`** — Mutable between calls. Reassign via `ts_heatmap.aggregation_method = ...` before calling `aggregate_scores` again.
- **`grouping`** — Mutable between calls. Reassign via `ts_heatmap.grouping = [...]` to switch between grouping dimensions (e.g. `['sector', 'region']`, `['sector']`) without reinstantiating.

The `analysis_parameters` tuple passed to `plot_grouped_heatmap` and `get_contributions_per_group` selects which single time frame and scope to visualise from the computed `grouped_aggregations`.

In [ ]:
# General instance — set time_frames and scopes broadly upfront.
# grouping and aggregation_method can be reassigned between calls to aggregate_scores.
ts_heatmap = TemperatureScore(
    time_frames=[ETimeFrames.SHORT, ETimeFrames.MID, ETimeFrames.LONG],
    scopes=[EScope.S1, EScope.S2, EScope.S1S2, EScope.S3, EScope.S1S2S3],
    aggregation_method=PortfolioAggregationMethod.WATS,
)

In [ ]:
grouping = ['industry_level_2', 'country']
ts_heatmap.grouping = grouping
grouped_aggregations = ts_heatmap.aggregate_scores(df_portfolio)

In [ ]:
analysis_parameters = ([ETimeFrames.MID], [EScope.S1S2], grouping)
timeframe_str = str(analysis_parameters[0][0]).lower()
scope_str = str(analysis_parameters[1][0])
method_str = ts_heatmap.aggregation_method.value

print(f"Scope: {scope_str} | Time frame: {timeframe_str} | Method: {method_str}")
plot_grouped_heatmap(grouped_aggregations, analysis_parameters)

# Build a pivot table matching the heatmap layout — select all and copy into Excel
aggregations = grouped_aggregations[timeframe_str][scope_str].grouped

rows = []
for key, agg in aggregations.items():
    g1_val, g2_val = key.split('|||', 1)
    rows.append({grouping[0]: g1_val, grouping[1]: g2_val, 'score': round(agg.score, 2)})

df_heatmap_table = pd.DataFrame(rows)
df_heatmap_pivot = df_heatmap_table.pivot(index=grouping[0], columns=grouping[1], values='score')
df_heatmap_pivot.columns.name = None

with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(df_heatmap_pivot)


#### Dive deeper into selected region and industry

In [ ]:
region = 'Europe'
sector = 'Utilities'
group = sector + '-' + region
analysis_parameters = ([ETimeFrames.MID], [EScope.S1S2], grouping)
timeframe_str = str(analysis_parameters[0][0]).lower()
scope_str = str(analysis_parameters[1][0])
method_str = ts_heatmap.aggregation_method.value

print(f"Scope: {scope_str} | Time frame: {timeframe_str} | Method: {method_str}")
group_contributions = get_contributions_per_group(grouped_aggregations, analysis_parameters, group)
group_contributions['scope'] = scope_str
group_contributions['timeframe'] = timeframe_str
group_contributions['method'] = method_str
display(group_contributions)

### Filtered drill-down: analysis within a subset

Filter the portfolio to a subset (e.g. a single region or country) and then run a grouped analysis on one or two other columns within that subset.

Configure the variables in the cell below:
- **`filter_col`** / **`filter_val`** — the column and value used to select the subset (e.g. `'region'` / `'Europe'`)
- **`grouping_drilldown`** — one or two columns to group by within the filtered subset (e.g. `['sector']` or `['sector', 'industry_level_1']`). With two columns a heatmap is shown; with one column only the table is produced.

The same `ts_heatmap` instance is reused — `grouping` is simply reassigned before calling `aggregate_scores`.


In [ ]:
# --- Configure the filter and grouping for the drill-down ---
# filter_col: any column in your portfolio data (e.g. 'region', 'country', 'sector')
# filter_val: the value to keep (e.g. 'Europe', 'Utilities')
filter_col = 'industry_level_1'
filter_val = 'Industrials'

# One or two grouping dimensions to analyse within the filtered subset.
# With two columns a heatmap is shown in addition to the table; with one column only the table is produced.
grouping_drilldown = ['country', 'industry_level_3']  # Can be any two columns in your portfolio data (e.g. 'sector' & 'industry_level_1')

# Select which time frame and scope to visualise
analysis_parameters_drilldown = ([ETimeFrames.MID], [EScope.S1S2], grouping_drilldown)


In [ ]:
df_filtered = df_portfolio[df_portfolio[filter_col] == filter_val].copy()
print(f"Filtered to {len(df_filtered)} rows where {filter_col} == '{filter_val}'")

ts_heatmap.grouping = grouping_drilldown
filtered_aggregations = ts_heatmap.aggregate_scores(df_filtered)

timeframe_str_dd = str(analysis_parameters_drilldown[0][0]).lower()
scope_str_dd = str(analysis_parameters_drilldown[1][0])
method_str_dd = ts_heatmap.aggregation_method.value

print(f"Scope: {scope_str_dd} | Time frame: {timeframe_str_dd} | Method: {method_str_dd} | Filter: {filter_col} = '{filter_val}'")

if len(grouping_drilldown) == 2:
    plot_grouped_heatmap(filtered_aggregations, analysis_parameters_drilldown)

# Build a tabular version of the same data — select all and copy into Excel
aggregations_dd = filtered_aggregations[timeframe_str_dd][scope_str_dd].grouped

rows_dd = []
for key, agg in aggregations_dd.items():
    row = {'score': round(agg.score, 2)}
    if len(grouping_drilldown) == 2:
        g1_val, g2_val = key.split('|||', 1)
        row[grouping_drilldown[0]] = g1_val
        row[grouping_drilldown[1]] = g2_val
    else:
        row[grouping_drilldown[0]] = key
    rows_dd.append(row)


df_drilldown_table = pd.DataFrame(rows_dd).sort_values(grouping_drilldown).reset_index(drop=True)

if len(grouping_drilldown) == 2:
    df_drilldown_pivot = df_drilldown_table.pivot(
        index=grouping_drilldown[1], columns=grouping_drilldown[0], values='score'
    )
    df_drilldown_pivot.columns.name = None
    with pd.option_context('display.max_rows', None, 'display.max_columns', None):
        display(df_drilldown_pivot)
else:
    with pd.option_context('display.max_rows', None):
        display(df_drilldown_table[[grouping_drilldown[0], 'score']])
